# Validate Variable Native Units and Unit Conversions
Checks that:
1. Each variable's native unit matches what the `climate_state_finder` notebook assumes
2. Each listed unit conversion succeeds (result `units` attribute matches the requested target)
3. Diagnoses which variables return `None` from `.get()` when `convert_units` is passed

In [ ]:
import climakitae as ck
import pandas as pd

cd = ck.ClimateData(verbosity=-1)
cd.table_id("mon")
cd.grid_label("d03")

## Define variables and expected conversions

In [ ]:
TO_TEST = [
    # (activity_id, institution_id, variable_id, expected_native_unit, [unit_options_to_test])
    ("WRF",   "UCLA", "prec",       "mm",           ["mm", "inches"]),
    ("LOCA2", "UCSD", "pr",         "kg m-2 s-1",   ["kg m-2 s-1", "mm", "inches"]),
    ("WRF",   "UCLA", "rh",         "%[0 to 100]",  ["%[0 to 100]"]),
    ("LOCA2", "UCSD", "hursmax",    "percent",      ["percent"]),
    ("LOCA2", "UCSD", "hursmin",    "percent",      ["percent"]),
    ("WRF",   "UCLA", "t2",         "K",            ["K", "degC", "degF"]),
    ("WRF",   "UCLA", "t2max",      "K",            ["K", "degC", "degF"]),
    ("WRF",   "UCLA", "t2min",      "K",            ["K", "degC", "degF"]),
    ("LOCA2", "UCSD", "tasmax",     "K",            ["K", "degC", "degF"]),
    ("LOCA2", "UCSD", "tasmin",     "K",            ["K", "degC", "degF"]),
    ("WRF",   "UCLA", "sw_sfc",     "W/m2",         ["W/m2"]),
    ("LOCA2", "UCSD", "rsds",       "W m-2",        ["W m-2"]),
    ("WRF",   "UCLA", "q2",         "g/kg",         ["g/kg", "kg/kg"]),
    ("LOCA2", "UCSD", "huss",       "1",            ["1"]),
    ("WRF",   "UCLA", "wspd10mean", "m s-1",        ["m s-1", "mph", "knots"]),
    ("LOCA2", "UCSD", "wspeed",     "m s-1",        ["m s-1", "mph", "knots"]),
    ("LOCA2", "UCSD", "uas",        "m s-1",        ["m s-1", "mph", "knots"]),
    ("LOCA2", "UCSD", "vas",        "m s-1",        ["m s-1", "mph", "knots"]),
]

## 1. Check native units
Retrieve each variable with no unit conversion and check the `units` attribute.

In [ ]:
native_results = []

for activity, institution, var, expected_native, _ in TO_TEST:
    try:
        ds = (
            cd.catalog("cadcat")
            .activity_id(activity)
            .institution_id(institution)
            .table_id("mon")
            .grid_label("d03")
            .variable(var)
            .processes({"warming_level": {"warming_levels": [0.8], "add_dummy_time": True}})
        ).get()
        if ds is None:
            actual_native = "ERROR: .get() returned None"
            match = False
        else:
            actual_native = ds[var].attrs.get("units", "NOT FOUND")
            match = actual_native == expected_native
    except Exception as e:
        actual_native = f"ERROR: {e}"
        match = False

    native_results.append({
        "activity": activity,
        "variable": var,
        "expected_native": expected_native,
        "actual_native": actual_native,
        "match": "✓" if match else "✗",
    })

pd.DataFrame(native_results)

## 2. Diagnose dataset keys for each variable
For variables where `ds[var]` was failing, check what keys are actually in the returned dataset.

In [ ]:
key_results = []

for activity, institution, var, _, _ in TO_TEST:
    try:
        ds = (
            cd.catalog("cadcat")
            .activity_id(activity)
            .institution_id(institution)
            .table_id("mon")
            .grid_label("d03")
            .variable(var)
            .processes({"warming_level": {"warming_levels": [0.8], "add_dummy_time": True}})
        ).get()
        data_vars = list(ds.data_vars) if ds is not None else "None returned"
        key_in_ds = var in ds.data_vars if ds is not None else False
    except Exception as e:
        data_vars = f"ERROR: {e}"
        key_in_ds = False

    key_results.append({
        "activity": activity,
        "variable": var,
        "data_vars": data_vars,
        "var_key_present": "✓" if key_in_ds else "✗",
    })

pd.DataFrame(key_results)

## 3. Check unit conversions
For each variable and each listed unit option, retrieve with `convert_units` and verify the result's `units` attribute matches the requested target. A mismatch means the conversion silently failed.

In [ ]:
conversion_results = []

for activity, institution, var, _, unit_options in TO_TEST:
    for target_unit in unit_options:
        try:
            ds = (
                cd.catalog("cadcat")
                .activity_id(activity)
                .institution_id(institution)
                .table_id("mon")
                .grid_label("d03")
                .variable(var)
                .processes({
                    "warming_level": {"warming_levels": [0.8], "add_dummy_time": True},
                    "convert_units": target_unit,
                })
            ).get()
            if ds is None:
                result_unit = ".get() returned None"
                match = False
            else:
                result_unit = ds[var].attrs.get("units", "NOT FOUND")
                match = result_unit == target_unit
        except Exception as e:
            result_unit = f"ERROR: {e}"
            match = False

        conversion_results.append({
            "activity": activity,
            "variable": var,
            "target_unit": target_unit,
            "result_unit": result_unit,
            "match": "✓" if match else "✗",
        })

pd.DataFrame(conversion_results)